# Data Cleaning - VAERSDATA.csv

In [54]:
# import libraries
import pandas as pd
import numpy as np
import scipy.stats as sci
import datetime
import kagglehub as kh
from kagglehub import KaggleDatasetAdapter

# Ignore TqdmWarning error message; shouldn't pose a problem

## Read in + inspect VAERS data

In [55]:
# Download latest csv version
path = kh.dataset_download("ayushggarg/covid19-vaccine-adverse-reactions")
print("Path to dataset files:", path)

Path to dataset files: /home/ekmys/.cache/kagglehub/datasets/ayushggarg/covid19-vaccine-adverse-reactions/versions/7


In [56]:
# create dataframe for df_VAERS
path_to_VAERS = path + '/VAERSDATA.csv'
df_VAERS = pd.read_csv(path_to_VAERS)
df_VAERS.head()

/tmp/ipykernel_26625/3462655922.py:3: DtypeWarning: Columns (7,12,15,23) have mixed types. Specify dtype option on import or set low_memory=False.
  df_VAERS = pd.read_csv(path_to_VAERS)


,VAERS_ID,RECVDATE,STATE,AGE_YRS,CAGE_YR,CAGE_MO,SEX,RPT_DATE,SYMPTOM_TEXT,DIED,...,CUR_ILL,HISTORY,PRIOR_VAX,SPLTTYPE,FORM_VERS,TODAYS_DATE,BIRTH_DEFECT,OFC_VISIT,ER_ED_VISIT,ALLERGIES
0,902418,12/15/2020,NJ,56.0,56.0,NaN,F,NaN,Patient experienced mild numbness traveling fr...,NaN,...,none,none,NaN,NaN,2,12/15/2020,NaN,NaN,NaN,none
1,902440,12/15/2020,AZ,35.0,35.0,NaN,F,NaN,C/O Headache,NaN,...,NaN,NaN,NaN,NaN,2,12/15/2020,NaN,NaN,NaN,NaN
2,902446,12/15/2020,WV,55.0,55.0,NaN,F,NaN,"felt warm, hot and face and ears were red and ...",NaN,...,none,"Hypertension, sleep apnea, hypothyroidism",NaN,NaN,2,12/15/2020,NaN,NaN,NaN,"Contrast Dye IV contrast, shellfish, strawberry"
3,902464,12/15/2020,LA,42.0,42.0,NaN,M,NaN,within 15 minutes progressive light-headedness...,NaN,...,none,none,NaN,NaN,2,12/15/2020,NaN,NaN,Y,none
4,902465,12/15/2020,AR,60.0,60.0,NaN,F,NaN,Pt felt wave come over body @ 1218 starting in...,NaN,...,"Bronchitis, finished prednisone on 12-13-20","hypertension, fibromyalgia",NaN,NaN,2,12/15/2020,NaN,NaN,NaN,Biaxin


In [57]:
# inspect shape & info for VAERS data
df_VAERS.info()
print("Shape of df_VAERS: " , df_VAERS.shape)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1012894 entries, 0 to 1012893
Data columns (total 35 columns):
 #   Column        Non-Null Count    Dtype  
---  ------        --------------    -----  
 0   VAERS_ID      1012894 non-null  int64  
 1   RECVDATE      1012894 non-null  object 
 2   STATE         842425 non-null   object 
 3   AGE_YRS       909608 non-null   float64
 4   CAGE_YR       809314 non-null   float64
 5   CAGE_MO       5374 non-null     float64
 6   SEX           1012894 non-null  object 
 7   RPT_DATE      1130 non-null     object 
 8   SYMPTOM_TEXT  1011423 non-null  object 
 9   DIED          18951 non-null    object 
 10  DATEDIED      16828 non-null    object 
 11  L_THREAT      15197 non-null    object 
 12  ER_VISIT      144 non-null      object 
 13  HOSPITAL      90081 non-null    object 
 14  HOSPDAYS      53040 non-null    float64
 15  X_STAY        505 non-null      object 
 16  DISABLE       18274 non-null    object 
 17  RECOVD        882224 non-nu

In [58]:
# summary statistics
df_VAERS.describe().T

,count,mean,std,min,25%,50%,75%,max
VAERS_ID,1012894.0,1.702870e+06,543887.999638,902418.00,1220910.25,1621311.5,2145853.75,2776336.0
AGE_YRS,909608.0,5.009099e+01,20.284642,0.08,35.00,51.0,66.00,119.0
CAGE_YR,809314.0,4.971534e+01,20.574754,0.00,34.00,51.0,66.00,120.0
CAGE_MO,5374.0,3.620767e-01,0.329444,0.00,0.00,0.3,0.60,1.0
HOSPDAYS,53040.0,1.027675e+01,616.555189,1.00,2.00,4.0,7.00,99999.0
NUMDAYS,876293.0,4.437788e+01,608.346874,0.00,0.00,1.0,8.00,66108.0
FORM_VERS,1012894.0,1.998811e+00,0.034457,1.00,2.00,2.0,2.00,2.0


In [59]:
# identify any duplicate VAERS_ID values
duplicates_VAERSDATA_df = df_VAERS[df_VAERS.duplicated(['VAERS_ID'])]
display(duplicates_VAERSDATA_df)

# Filter to show only duplicated names
duplicates_VAERSDATA_df1 = df_VAERS['VAERS_ID'].value_counts()[df_VAERS['VAERS_ID'].value_counts() > 1]
print("Number of duplicate VAERS IDs in VAERS dataset: ", len(duplicates_VAERSDATA_df1))

# Code example from: https://pythonguides.com/how-to-find-duplicates-in-python-dataframe/

,VAERS_ID,RECVDATE,STATE,AGE_YRS,CAGE_YR,CAGE_MO,SEX,RPT_DATE,SYMPTOM_TEXT,DIED,...,CUR_ILL,HISTORY,PRIOR_VAX,SPLTTYPE,FORM_VERS,TODAYS_DATE,BIRTH_DEFECT,OFC_VISIT,ER_ED_VISIT,ALLERGIES


Number of duplicate VAERS IDs in VAERS dataset:  0


**Inspection Summary**
- df_VAERS contains summary data for each adverse event, including patient age, location, and pertinent medical data; dates that the report was received, vaccine was administered, onset of the adverse event, and date of patient death if relevant; data on the patient's treatment if treated at the ER and/or hospitalized; text description of the patient's symptoms; and administrative data on the facility administering the vaccine, manufacturer project report number, and any diagnostic laboratory data.
- Shape: df_VAERS contains 35 columns and 1,012,894 rows
- A majority of columns have missing data / null values
- Columns without missing data / null values include VAERS_ID, RECVDATE (recovery date), SEX, V_ADMINBY (type of facility where vaccine was administered), and FORM_VERS (which version, 1 or 2, of the VAERS form was submitted)
- There are no duplicate VAERS ID's in the VAERS dataset.

# Remove Whitespace

In [60]:
# remove whitespace at head & tail of string-type columns
for column in df_VAERS.columns:
    if df_VAERS[column].dtype == 'str':
        df_VAERS[column] = df_VAERS[column].str.strip()

# remove whitespace from any single-word columns
single_word_cols_VAERS = list(df_VAERS[['RECVDATE','STATE','RPT_DATE','DIED','DATEDIED','L_THREAT','ER_VISIT','HOSPITAL','X_STAY','DISABLE','BIRTH_DEFECT','RECOVD','VAX_DATE','ONSET_DATE','V_ADMINBY','V_FUNDBY']])
for col in single_word_cols_VAERS:
    df_VAERS[col] = df_VAERS[col].replace(" ","")

# Ensure numeric columns contain only Int64 or Float type

In [61]:
# ensure all columns that should be number columns only contain numbers
for column in df_VAERS.columns:
    if df_VAERS[column].dtype == 'int64':
        df_VAERS[column] = df_VAERS[column].astype('Int64')

for column in df_VAERS.columns:
    if df_VAERS[column].dtype == 'Float':
        df_VAERS[column] = df_VAERS[column].astype('Float')

# Ensure categorical data contains valid entries

In [62]:
# get unique values for relevant columns
unique_STATE = list(df_VAERS['STATE'].unique())
unique_SEX = list(df_VAERS['SEX'].unique())
unique_V_ADMINBY = list(df_VAERS['V_ADMINBY'].unique())
unique_V_FUNDBY = list(df_VAERS['V_FUNDBY'].unique())
unique_RECOVD = list(df_VAERS['RECOVD'].unique())
unique_FORM_VERS = list(df_VAERS['FORM_VERS'].unique())

# unique_STATE - remove non-string values
for item in unique_STATE:
    if type(item) != str:
        unique_STATE.remove(item)

unique_STATE = sorted(unique_STATE)

print("Unique values in 'STATE': ", unique_STATE)
print("Unique values in 'SEX': ", unique_SEX)
print("Unique values in 'V_ADMINBY': ", unique_V_ADMINBY)
print("Unique values in V_FUNDBY': ", unique_V_FUNDBY)
print("Unique values in 'RECOVD': ", unique_RECOVD)
print("Unique values in 'FORM_VERS': ", unique_FORM_VERS)

# Code from: https://www.geeksforgeeks.org/python/get-unique-values-from-a-column-in-pandas-dataframe/

Unique values in 'STATE':  ['AK', 'AL', 'AR', 'AS', 'AZ', 'CA', 'CO', 'CT', 'Ca', 'DC', 'DE', 'FL', 'FM', 'GA', 'GU', 'HI', 'IA', 'ID', 'IL', 'IN', 'KS', 'KY', 'LA', 'MA', 'MD', 'ME', 'MH', 'MI', 'MN', 'MO', 'MP', 'MS', 'MT', 'NC', 'ND', 'NE', 'NH', 'NJ', 'NM', 'NV', 'NY', 'OH', 'OK', 'OR', 'PA', 'PR', 'PW', 'QM', 'QW', 'RI', 'SC', 'SD', 'TN', 'TX', 'Tx', 'UT', 'VA', 'VI', 'VT', 'WA', 'WI', 'WV', 'WY', 'XB', 'XL', 'XV']
Unique values in 'SEX':  ['F', 'M', 'U']
Unique values in 'V_ADMINBY':  ['PVT', 'OTH', 'PUB', 'WRK', 'UNK', 'PHM', 'MIL', 'SEN', 'SCH']
Unique values in V_FUNDBY':  [nan, 'OTH', 'UNK', 'PUB', 'PVT', 'MIL']
Unique values in 'RECOVD':  ['Y', 'N', 'U', nan]
Unique values in 'FORM_VERS':  [np.int64(2), np.int64(1)]


**df_VAERS Labeled Data Issues:**

STATE
- 'FM', 'MH', 'PW', 'QM', 'QW', 'XB', 'XL', 'XV' - typos?
- 'Tx' can likely assume is meant to be 'TX'
- 'Ca' can likely assume is meant to be 'CA'

- Note - AS: American Samoa, GU: Guam, Northern Mariana Islands: MP, Puerto Rico: PR, Trust Territories: TT, Virgin Islands: VI (https://www.faa.gov/air_traffic/publications/atpubs/cnt_html/appendix_a.html)

SEX - no issues (F: Female, M: Male, U: Unknown)

V_ADMINBY - no issues (VAERS 1.0: PUB=Public, PVT=Private, MIL=Military, OTH=Other, UNK=Unknown;
VAERS 2.0: PUB=Public, PVT=Private, MIL=Military, PHM=Pharmacy or store, SCH=School or student health clinic, SEN=Nursing home or senior living facility, WRK=Workplace clinic, OTH=Other, UNK=Unknown) (User Guide, p. 11)

V_FUNDBY - no issues; same acronyms as V_ADMINBY

RECOVD - no issues (Y: Yes, N: No, U: Unknown)

FORM_VERS - no issues; either 1 or 2 (Form Version 1 or Form Version 2)

In [63]:
# fix issues in STATE column
df_VAERS['STATE'] = df_VAERS['STATE'].replace('Tx','TX')
df_VAERS['STATE'] = df_VAERS['STATE'].replace('Ca','CA')

# count instances of random typos to make sure they're not anything specific
FM = (df_VAERS['STATE'] == 'FM').sum()
MH = (df_VAERS['STATE'] == 'MH').sum()
PW = (df_VAERS['STATE'] == 'PW').sum()
QM = (df_VAERS['STATE'] == 'QM').sum()
QW = (df_VAERS['STATE'] == 'QW').sum()
XB = (df_VAERS['STATE'] == 'XB').sum()
XL = (df_VAERS['STATE'] == 'XL').sum()
XV = (df_VAERS['STATE'] == 'XV').sum()

print("Occurrences of 'FM':", FM)
print("Occurrences of 'MH':", MH)
print("Occurrences of 'PW':", PW)
print("Occurrences of 'QM':", QM)
print("Occurrences of 'QW':", QW)
print("Occurrences of 'XB':", XB)
print("Occurrences of 'XL':", XL)
print("Occurrences of 'XV':", XV)

'''None of the typos have higher than 10 instances. Replace with NaN'''

Occurrences of 'FM': 8
Occurrences of 'MH': 8
Occurrences of 'PW': 2
Occurrences of 'QM': 3
Occurrences of 'QW': 2
Occurrences of 'XB': 10
Occurrences of 'XL': 1
Occurrences of 'XV': 2


'None of the typos have higher than 10 instances. Replace with NaN'

In [64]:
# replacing remaining STATE typos
df_VAERS['STATE'] = df_VAERS['STATE'].replace(['FM', 'MH', 'PW', 'QM', 'QW', 'XB', 'XL', 'XV'], np.nan)

# Convert date columns to date object

In [65]:
# convert date columns to datetime objects
df_VAERS['RECVDATE'] = df_VAERS['RECVDATE'].astype('datetime64[ns]')
df_VAERS['DATEDIED'] = df_VAERS['DATEDIED'].astype('datetime64[ns]')
df_VAERS['VAX_DATE'] = df_VAERS['VAX_DATE'].astype('datetime64[ns]')
df_VAERS['ONSET_DATE'] = df_VAERS['ONSET_DATE'].astype('datetime64[ns]')
df_VAERS['TODAYS_DATE'] = df_VAERS['TODAYS_DATE'].astype('datetime64[ns]')
df_VAERS['RPT_DATE'] = df_VAERS['RPT_DATE'].astype('datetime64[ns]')

# verify conversion to datetime object
print(df_VAERS['RECVDATE'].dtype)
print(df_VAERS['DATEDIED'].dtype)
print(df_VAERS['VAX_DATE'].dtype)
print(df_VAERS['ONSET_DATE'].dtype)
print(df_VAERS['TODAYS_DATE'].dtype)
print(df_VAERS['RPT_DATE'].dtype)

datetime64[ns]
datetime64[ns]
datetime64[ns]
datetime64[ns]
datetime64[ns]
datetime64[ns]


# Add year, month, etc. columns

In [66]:
# add more date objects to dataframe
df_VAERS['ONSET_YEAR'] = df_VAERS['ONSET_DATE'].dt.year
df_VAERS['ONSET_MONTH'] = df_VAERS['ONSET_DATE'].dt.strftime('%b')
df_VAERS['ONSET_MONTHYEAR'] = df_VAERS['ONSET_DATE'].dt.strftime('%Y-%m')
df_VAERS.head()

,VAERS_ID,RECVDATE,STATE,AGE_YRS,CAGE_YR,CAGE_MO,SEX,RPT_DATE,SYMPTOM_TEXT,DIED,...,SPLTTYPE,FORM_VERS,TODAYS_DATE,BIRTH_DEFECT,OFC_VISIT,ER_ED_VISIT,ALLERGIES,ONSET_YEAR,ONSET_MONTH,ONSET_MONTHYEAR
0,902418,2020-12-15,NJ,56.0,56.0,NaN,F,NaT,Patient experienced mild numbness traveling fr...,NaN,...,NaN,2,2020-12-15,NaN,NaN,NaN,none,2020.0,Dec,2020-12
1,902440,2020-12-15,AZ,35.0,35.0,NaN,F,NaT,C/O Headache,NaN,...,NaN,2,2020-12-15,NaN,NaN,NaN,NaN,2020.0,Dec,2020-12
2,902446,2020-12-15,WV,55.0,55.0,NaN,F,NaT,"felt warm, hot and face and ears were red and ...",NaN,...,NaN,2,2020-12-15,NaN,NaN,NaN,"Contrast Dye IV contrast, shellfish, strawberry",2020.0,Dec,2020-12
3,902464,2020-12-15,LA,42.0,42.0,NaN,M,NaT,within 15 minutes progressive light-headedness...,NaN,...,NaN,2,2020-12-15,NaN,NaN,Y,none,2020.0,Dec,2020-12
4,902465,2020-12-15,AR,60.0,60.0,NaN,F,NaT,Pt felt wave come over body @ 1218 starting in...,NaN,...,NaN,2,2020-12-15,NaN,NaN,NaN,Biaxin,2020.0,Dec,2020-12


# Transform string columns with binary (Y/N) values

In [67]:
# get counts of unique values in binary-value string columns
unique_DIED = list(df_VAERS['DIED'].unique())
unique_L_THREAT = list(df_VAERS['L_THREAT'].unique())
unique_ER_VISIT = list(df_VAERS['ER_VISIT'].unique())
unique_HOSPITAL = list(df_VAERS['HOSPITAL'].unique())
unique_X_STAY = list(df_VAERS['X_STAY'].unique())
unique_DISABLE = list(df_VAERS['DISABLE'].unique())
unique_BIRTH_DEFECT = list(df_VAERS['BIRTH_DEFECT'].unique())

print("Unique values in 'DIED': ", unique_DIED)
print("Unique values in 'L_THREAT': ", unique_L_THREAT)
print("Unique values in 'ER_VISIT': ", unique_ER_VISIT)
print("Unique values in 'HOSPITAL': ", unique_HOSPITAL)
print("Unique values in 'X_STAY': ", unique_X_STAY)
print("Unique values in 'DISABLE': ", unique_DISABLE)
print("Unique values in 'BIRTH_DEFECT': ", unique_BIRTH_DEFECT)

Unique values in 'DIED':  [nan, 'Y']
Unique values in 'L_THREAT':  [nan, 'Y']
Unique values in 'ER_VISIT':  [nan, 'Y']
Unique values in 'HOSPITAL':  [nan, 'Y']
Unique values in 'X_STAY':  [nan, 'Y']
Unique values in 'DISABLE':  [nan, 'Y']
Unique values in 'BIRTH_DEFECT':  [nan, 'Y']


In [68]:
# df_VAERS - change nan values to 'N' in boolean columns (string data type with binary values)
df_VAERS['DIED'] = df_VAERS['DIED'].replace(np.nan,'N')
df_VAERS['L_THREAT'] = df_VAERS['L_THREAT'].replace(np.nan,'N')
df_VAERS['ER_VISIT'] = df_VAERS['ER_VISIT'].replace(np.nan,'N')
df_VAERS['HOSPITAL'] = df_VAERS['HOSPITAL'].replace(np.nan,'N')
df_VAERS['X_STAY'] = df_VAERS['X_STAY'].replace(np.nan,'N')
df_VAERS['DISABLE'] = df_VAERS['DISABLE'].replace(np.nan,'N')
df_VAERS['BIRTH_DEFECT'] = df_VAERS['BIRTH_DEFECT'].replace(np.nan,'N')

'''Reasoning:
The columns 'DIED', 'L_THREAT', 'ER_VISIT', 'HOSPITAL', 'X_STAY', 'DISABLE', and 'BIRTH_DEFECT', are all string columns with binary values indicating patient outcomes. (I.e. whether the patient died, experienced a life-threatening emergency, visited the ER, was hospitalized, required an extended stay in the hospital, or became disabled as part of their adverse vaccine reaction, or had a congenital birth defect associated with the vaccine.) All of these columns either have the value 'Y' for "Yes" or are blank (nan values). I think it is fair to say that changing nan to 'N' for "No" can be interpreted to mean that the VAERS report did not confirm that the patient experienced the outcomes in these boolean columns.

Note - The 'RECOVD' column ("Recovered") has 3 values: 'Y', 'N', or 'U' for Unknown, so I'm not including it in this transformation.'''

'Reasoning:\nThe columns \'DIED\', \'L_THREAT\', \'ER_VISIT\', \'HOSPITAL\', \'X_STAY\', \'DISABLE\', and \'BIRTH_DEFECT\', are all string columns with binary values indicating patient outcomes. (I.e. whether the patient died, experienced a life-threatening emergency, visited the ER, was hospitalized, required an extended stay in the hospital, or became disabled as part of their adverse vaccine reaction, or had a congenital birth defect associated with the vaccine.) All of these columns either have the value \'Y\' for "Yes" or are blank (nan values). I think it is fair to say that changing nan to \'N\' for "No" can be interpreted to mean that the VAERS report did not confirm that the patient experienced the outcomes in these boolean columns.\n\nNote - The \'RECOVD\' column ("Recovered") has 3 values: \'Y\', \'N\', or \'U\' for Unknown, so I\'m not including it in this transformation.'

In [69]:
# verify transformation
unique_DIED = list(df_VAERS['DIED'].unique())
unique_L_THREAT = list(df_VAERS['L_THREAT'].unique())
unique_ER_VISIT = list(df_VAERS['ER_VISIT'].unique())
unique_HOSPITAL = list(df_VAERS['HOSPITAL'].unique())
unique_X_STAY = list(df_VAERS['X_STAY'].unique())
unique_DISABLE = list(df_VAERS['DISABLE'].unique())
unique_BIRTH_DEFECT = list(df_VAERS['BIRTH_DEFECT'].unique())

print("Unique values in 'DIED': ", unique_DIED)
print("Unique values in 'L_THREAT': ", unique_L_THREAT)
print("Unique values in 'ER_VISIT': ", unique_ER_VISIT)
print("Unique values in 'HOSPITAL': ", unique_HOSPITAL)
print("Unique values in 'X_STAY': ", unique_X_STAY)
print("Unique values in 'DISABLE': ", unique_DISABLE)
print("Unique values in 'BIRTH_DEFECT': ", unique_BIRTH_DEFECT)

Unique values in 'DIED':  ['N', 'Y']
Unique values in 'L_THREAT':  ['N', 'Y']
Unique values in 'ER_VISIT':  ['N', 'Y']
Unique values in 'HOSPITAL':  ['N', 'Y']
Unique values in 'X_STAY':  ['N', 'Y']
Unique values in 'DISABLE':  ['N', 'Y']
Unique values in 'BIRTH_DEFECT':  ['N', 'Y']


In [70]:
# change nan values to 'N' in remaining two Y/N columns for same reasoning as above
unique_OFC_VISIT = list(df_VAERS['OFC_VISIT'].unique())
unique_ER_ED_VISIT = list(df_VAERS['ER_ED_VISIT'].unique())

print(unique_OFC_VISIT)
print(unique_ER_ED_VISIT)

df_VAERS['OFC_VISIT'] = df_VAERS['OFC_VISIT'].replace(np.nan,'N')
df_VAERS['ER_ED_VISIT'] = df_VAERS['ER_ED_VISIT'].replace(np.nan,'N')

[nan, 'Y']
[nan, 'Y']


In [71]:
# verify transformation
unique_OFC_VISIT = list(df_VAERS['OFC_VISIT'].unique())
unique_ER_ED_VISIT = list(df_VAERS['ER_ED_VISIT'].unique())

print(unique_OFC_VISIT)
print(unique_ER_ED_VISIT)

['N', 'Y']
['N', 'Y']


# Drop Unnecessary Columns

In [72]:
df_VAERS_filtered = df_VAERS.drop(['CAGE_YR', 'CAGE_MO', 'SYMPTOM_TEXT', 'LAB_DATA', 'OTHER_MEDS', 'CUR_ILL', 'HISTORY', 'PRIOR_VAX', 'SPLTTYPE', 'ALLERGIES'], axis=1)

In [73]:
df_VAERS_filtered.head()

,VAERS_ID,RECVDATE,STATE,AGE_YRS,SEX,RPT_DATE,DIED,DATEDIED,L_THREAT,ER_VISIT,...,V_ADMINBY,V_FUNDBY,FORM_VERS,TODAYS_DATE,BIRTH_DEFECT,OFC_VISIT,ER_ED_VISIT,ONSET_YEAR,ONSET_MONTH,ONSET_MONTHYEAR
0,902418,2020-12-15,NJ,56.0,F,NaT,N,NaT,N,N,...,PVT,NaN,2,2020-12-15,N,N,N,2020.0,Dec,2020-12
1,902440,2020-12-15,AZ,35.0,F,NaT,N,NaT,N,N,...,PVT,NaN,2,2020-12-15,N,N,N,2020.0,Dec,2020-12
2,902446,2020-12-15,WV,55.0,F,NaT,N,NaT,N,N,...,OTH,NaN,2,2020-12-15,N,N,N,2020.0,Dec,2020-12
3,902464,2020-12-15,LA,42.0,M,NaT,N,NaT,N,N,...,PVT,NaN,2,2020-12-15,N,N,Y,2020.0,Dec,2020-12
4,902465,2020-12-15,AR,60.0,F,NaT,N,NaT,N,N,...,PUB,NaN,2,2020-12-15,N,N,N,2020.0,Dec,2020-12


**Reasoning for dropping the columns above:**
These columns contain narrative data (i.e. not from controlled lists) about patient circumstances and will require more in-depth parsing and data cleaning to make useful, beyond the scope of an initial EDA.

In [74]:
df_VAERS_filtered.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1012894 entries, 0 to 1012893
Data columns (total 28 columns):
 #   Column           Non-Null Count    Dtype         
---  ------           --------------    -----         
 0   VAERS_ID         1012894 non-null  Int64         
 1   RECVDATE         1012894 non-null  datetime64[ns]
 2   STATE            842389 non-null   object        
 3   AGE_YRS          909608 non-null   float64       
 4   SEX              1012894 non-null  object        
 5   RPT_DATE         1130 non-null     datetime64[ns]
 6   DIED             1012894 non-null  object        
 7   DATEDIED         16828 non-null    datetime64[ns]
 8   L_THREAT         1012894 non-null  object        
 9   ER_VISIT         1012894 non-null  object        
 10  HOSPITAL         1012894 non-null  object        
 11  HOSPDAYS         53040 non-null    float64       
 12  X_STAY           1012894 non-null  object        
 13  DISABLE          1012894 non-null  object        
 14  RE

# Replace null values with "UNK" ("Unknown") in 'RECOVD', 'V_ADMINBY', 'V_FUNDBY'

In [75]:
## Code from Brian Zavala's EDA (now that individual EDAs are turned in and we're collaborating as a group)

## Keeping as string, specific cleaning instructions regarding NULLS provided by data dictionary
df_VAERS_filtered.fillna({'RECOVD':"U"}, inplace = True) # Data of unknown status of vaccine recovery is in column already. Just replacing NULLS with "U"
df_VAERS_filtered.fillna({'V_ADMINBY':"UNK"}, inplace = True) # Location of administration specifies an "Unknown" observation. Just replacing NULLS with "UNK"
df_VAERS_filtered.fillna({'V_FUNDBY':"UNK"}, inplace = True) # Specifies that purchase history can be Other/Unknown. Just replacing NULLS with "UNK"

## New functions learned and other comments.
# .notnull() https://pandas.pydata.org/docs/reference/api/pandas.notnull.html
# .astype(bool) is stupid https://www.reddit.com/r/learnpython/comments/1mokeww/the_insidious_astypebool/

# Handle Outliers

In [76]:
## Code from Brian Zavala's EDA (now that individual EDAs are turned in and we're collaborating as a group)

# IRQ Functions for outliers - created these during lab_02_todo
def get_iqr_upper(x):
    return x.quantile(0.75)+(1.5*(x.quantile(0.75)-x.quantile(0.25)))

def get_iqr_lower(x):
    iqr_low = x.quantile(0.25)-(1.5*(x.quantile(0.75)-x.quantile(0.25)))
    if iqr_low < 0:
        return 0
    else:
        return iqr_low

# adding lower range function for date handling
def get_iqr_lower_date(x):
    return x.quantile(0.25)-(1.5*(x.quantile(0.75)-x.quantile(0.25)))

In [77]:
# summary statistics
df_VAERS_filtered.describe().T

,count,mean,min,25%,50%,75%,max,std
VAERS_ID,1012894.0,1702869.518144,902418.0,1220910.25,1621311.5,2145853.75,2776336.0,543887.999638
RECVDATE,1012894,2021-10-20 20:11:48.389229312,2020-12-15 00:00:00,2021-04-12 00:00:00,2021-08-21 00:00:00,2022-02-04 00:00:00,2024-06-28 00:00:00,NaN
AGE_YRS,909608.0,50.090985,0.08,35.0,51.0,66.0,119.0,20.284642
RPT_DATE,1130,2021-06-15 14:44:23.362831616,2020-01-13 00:00:00,2021-01-11 00:00:00,2021-02-25 00:00:00,2021-08-20 00:00:00,2024-05-21 00:00:00,NaN
DATEDIED,16828,2021-10-13 17:37:50.216305664,1921-03-14 00:00:00,2021-04-13 00:00:00,2021-09-14 00:00:00,2022-01-28 00:00:00,2024-06-22 00:00:00,NaN
HOSPDAYS,53040.0,10.276753,1.0,2.0,4.0,7.0,99999.0,616.555189
VAX_DATE,938970,2021-06-23 07:17:13.957634560,1900-02-01 00:00:00,2021-02-21 00:00:00,2021-04-12 00:00:00,2021-10-09 00:00:00,2024-06-27 00:00:00,NaN
ONSET_DATE,915481,2021-07-28 18:34:00.162930688,1900-01-01 00:00:00,2021-03-04 00:00:00,2021-05-01 00:00:00,2021-11-12 00:00:00,2202-12-10 00:00:00,NaN
NUMDAYS,876293.0,44.377878,0.0,0.0,1.0,8.0,66108.0,608.346874
FORM_VERS,1012894.0,1.998811,1.0,2.0,2.0,2.0,2.0,0.034457


## Handle Date Outliers

There are clear outliers in the date columns:
- Minimums of 1921 and 1900 in DATEDIED, VAX_DATE, and ONSET_DATE
- Max of 2202 in ONSET_DATE

In [78]:
# counts of rows with years before 2020
print("Counts of rows with years before 2020:")
print((df_VAERS_filtered['RECVDATE'].dt.year < 2020).sum())
print((df_VAERS_filtered['RPT_DATE'].dt.year < 2020).sum())
print((df_VAERS_filtered['DATEDIED'].dt.year < 2020).sum())
print((df_VAERS_filtered['VAX_DATE'].dt.year < 2020).sum())
print((df_VAERS_filtered['ONSET_DATE'].dt.year < 2020).sum())
print((df_VAERS_filtered['TODAYS_DATE'].dt.year < 2020).sum())

# counts of rows with years after 2025
print("Counts of rows with years after 2025:")
print((df_VAERS_filtered['RECVDATE'].dt.year > 2025).sum())
print((df_VAERS_filtered['RPT_DATE'].dt.year > 2025).sum())
print((df_VAERS_filtered['DATEDIED'].dt.year > 2025).sum())
print((df_VAERS_filtered['VAX_DATE'].dt.year > 2025).sum())
print((df_VAERS_filtered['ONSET_DATE'].dt.year > 2025).sum())
print((df_VAERS_filtered['TODAYS_DATE'].dt.year > 2025).sum())

print("Total length of df_VAERS_filtered: ", len(df_VAERS_filtered))

Counts of rows with years before 2020:
0
0
9
1241
420
32
Counts of rows with years after 2025:
0
0
0
0
1
0
Total length of df_VAERS_filtered:  1012894


In [79]:
# drop rows with dates before 2020 and after 2025

# set lower date limit
from datetime import datetime as dt
lower_date_str = '2020-01-01'
lower_date_limit = dt.strptime(lower_date_str, '%Y-%m-%d')

# set upper date limit
upper_date_str = '2025-12-31'
upper_date_limit = dt.strptime(upper_date_str, '%Y-%m-%d')

# verify
print(lower_date_limit)
print(upper_date_limit)
print(type(lower_date_limit))
print(type(upper_date_limit))

2020-01-01 00:00:00
2025-12-31 00:00:00
<class 'datetime.datetime'>
<class 'datetime.datetime'>


In [80]:
# drop rows with dates after 2025

# get index of row to drop (should only be one)
index_dates_above_2025 = (df_VAERS_filtered.index[df_VAERS_filtered['ONSET_DATE'].dt.year > 2025]).to_list()
print(index_dates_above_2025)
print(df_VAERS_filtered.iloc[947970])

# drop row
df_VAERS_filtered = df_VAERS_filtered.drop([947970])

[947970]
VAERS_ID                       2583431
RECVDATE           2023-02-17 00:00:00
STATE                               WA
AGE_YRS                           35.0
SEX                                  M
RPT_DATE                           NaT
DIED                                 N
DATEDIED                           NaT
L_THREAT                             N
ER_VISIT                             N
HOSPITAL                             N
HOSPDAYS                           NaN
X_STAY                               N
DISABLE                              N
RECOVD                               U
VAX_DATE           2021-12-10 00:00:00
ONSET_DATE         2202-12-10 00:00:00
NUMDAYS                        66108.0
V_ADMINBY                          UNK
V_FUNDBY                           UNK
FORM_VERS                            2
TODAYS_DATE        2023-02-17 00:00:00
BIRTH_DEFECT                         N
OFC_VISIT                            N
ER_ED_VISIT                          N
ONSET_YEAR      

In [81]:
# verify filter
df_VAERS_filtered.describe().T

,count,mean,min,25%,50%,75%,max,std
VAERS_ID,1012893.0,1702868.648791,902418.0,1220910.0,1621311.0,2145853.0,2776336.0,543887.564373
RECVDATE,1012893,2021-10-20 20:11:07.090403328,2020-12-15 00:00:00,2021-04-12 00:00:00,2021-08-21 00:00:00,2022-02-04 00:00:00,2024-06-28 00:00:00,NaN
AGE_YRS,909607.0,50.091002,0.08,35.0,51.0,66.0,119.0,20.284647
RPT_DATE,1130,2021-06-15 14:44:23.362831616,2020-01-13 00:00:00,2021-01-11 00:00:00,2021-02-25 00:00:00,2021-08-20 00:00:00,2024-05-21 00:00:00,NaN
DATEDIED,16828,2021-10-13 17:37:50.216305664,1921-03-14 00:00:00,2021-04-13 00:00:00,2021-09-14 00:00:00,2022-01-28 00:00:00,2024-06-22 00:00:00,NaN
HOSPDAYS,53040.0,10.276753,1.0,2.0,4.0,7.0,99999.0,616.555189
VAX_DATE,938969,2021-06-23 07:16:58.342885120,1900-02-01 00:00:00,2021-02-21 00:00:00,2021-04-12 00:00:00,2021-10-09 00:00:00,2024-06-27 00:00:00,NaN
ONSET_DATE,915480,2021-07-28 16:49:48.438851328,1900-01-01 00:00:00,2021-03-04 00:00:00,2021-05-01 00:00:00,2021-11-12 00:00:00,2024-06-27 00:00:00,NaN
NUMDAYS,876292.0,44.302488,0.0,0.0,1.0,8.0,44224.0,604.23985
FORM_VERS,1012893.0,1.998811,1.0,2.0,2.0,2.0,2.0,0.034457


In [82]:
# verify filter
print(df_VAERS_filtered.info())

<class 'pandas.core.frame.DataFrame'>
Index: 1012893 entries, 0 to 1012893
Data columns (total 28 columns):
 #   Column           Non-Null Count    Dtype         
---  ------           --------------    -----         
 0   VAERS_ID         1012893 non-null  Int64         
 1   RECVDATE         1012893 non-null  datetime64[ns]
 2   STATE            842388 non-null   object        
 3   AGE_YRS          909607 non-null   float64       
 4   SEX              1012893 non-null  object        
 5   RPT_DATE         1130 non-null     datetime64[ns]
 6   DIED             1012893 non-null  object        
 7   DATEDIED         16828 non-null    datetime64[ns]
 8   L_THREAT         1012893 non-null  object        
 9   ER_VISIT         1012893 non-null  object        
 10  HOSPITAL         1012893 non-null  object        
 11  HOSPDAYS         53040 non-null    float64       
 12  X_STAY           1012893 non-null  object        
 13  DISABLE          1012893 non-null  object        
 14  RECOVD 

In [83]:
# drop rows with dates before 2020

# get indices of rows to drop
index_dates_ONSET = (df_VAERS_filtered.index[df_VAERS_filtered['ONSET_DATE'].dt.year < 2020]).tolist()
index_dates_VAX = (df_VAERS_filtered.index[df_VAERS_filtered['VAX_DATE'].dt.year < 2020]).tolist()
index_dates_TODAY = (df_VAERS_filtered.index[df_VAERS_filtered['TODAYS_DATE'].dt.year < 2020]).tolist()
index_dates_DIED = (df_VAERS_filtered.index[df_VAERS_filtered['DATEDIED'].dt.year < 2020]).tolist()

index_dates_below_2020 = index_dates_ONSET + index_dates_VAX + index_dates_TODAY + index_dates_DIED

print(index_dates_below_2020[0:20])
print(df_VAERS_filtered.iloc[10109])

# drop rows
df_VAERS_filtered = df_VAERS_filtered.drop(index_dates_below_2020)

[10109, 13227, 15682, 16883, 17534, 17851, 18129, 18400, 19379, 26986, 27291, 28764, 30306, 30668, 39207, 39677, 40590, 40899, 41273, 43653]
VAERS_ID                        916310
RECVDATE           2020-12-31 00:00:00
STATE                               MA
AGE_YRS                           52.0
SEX                                  F
RPT_DATE                           NaT
DIED                                 N
DATEDIED                           NaT
L_THREAT                             N
ER_VISIT                             N
HOSPITAL                             N
HOSPDAYS                           NaN
X_STAY                               N
DISABLE                              N
RECOVD                               Y
VAX_DATE           2020-12-29 00:00:00
ONSET_DATE         1920-11-29 00:00:00
NUMDAYS                            NaN
V_ADMINBY                          OTH
V_FUNDBY                           UNK
FORM_VERS                            2
TODAYS_DATE        2020-12-31 00:00:00
B

In [84]:
# verify filter
df_VAERS_filtered.describe().T

,count,mean,min,25%,50%,75%,max,std
VAERS_ID,1011306.0,1702874.365239,902418.0,1220909.25,1621329.5,2145841.75,2776336.0,543899.884399
RECVDATE,1011306,2021-10-20 20:20:40.844611072,2020-12-15 00:00:00,2021-04-12 00:00:00,2021-08-21 00:00:00,2022-02-04 00:00:00,2024-06-28 00:00:00,NaN
AGE_YRS,908251.0,50.087924,0.08,35.0,51.0,66.0,119.0,20.284704
RPT_DATE,1128,2021-06-15 12:20:25.531913472,2020-01-13 00:00:00,2021-01-11 00:00:00,2021-02-25 00:00:00,2021-08-20 18:00:00,2024-05-21 00:00:00,NaN
DATEDIED,16801,2021-10-21 22:54:10.520802816,2020-01-10 00:00:00,2021-04-13 00:00:00,2021-09-14 00:00:00,2022-01-28 00:00:00,2024-06-22 00:00:00,NaN
HOSPDAYS,52962.0,10.168026,1.0,2.0,4.0,7.0,99999.0,616.448667
VAX_DATE,937409,2021-07-10 10:56:03.698022400,2020-01-01 00:00:00,2021-02-22 00:00:00,2021-04-12 00:00:00,2021-10-09 00:00:00,2024-06-27 00:00:00,NaN
ONSET_DATE,914081,2021-08-07 05:13:57.689878016,2020-01-01 00:00:00,2021-03-04 00:00:00,2021-05-01 00:00:00,2021-11-12 00:00:00,2024-06-27 00:00:00,NaN
NUMDAYS,875204.0,29.511665,0.0,0.0,1.0,8.0,1224.0,87.197297
FORM_VERS,1011306.0,1.998811,1.0,2.0,2.0,2.0,2.0,0.034455


In [85]:
# verify filter
df_VAERS_filtered.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1011306 entries, 0 to 1012893
Data columns (total 28 columns):
 #   Column           Non-Null Count    Dtype         
---  ------           --------------    -----         
 0   VAERS_ID         1011306 non-null  Int64         
 1   RECVDATE         1011306 non-null  datetime64[ns]
 2   STATE            841109 non-null   object        
 3   AGE_YRS          908251 non-null   float64       
 4   SEX              1011306 non-null  object        
 5   RPT_DATE         1128 non-null     datetime64[ns]
 6   DIED             1011306 non-null  object        
 7   DATEDIED         16801 non-null    datetime64[ns]
 8   L_THREAT         1011306 non-null  object        
 9   ER_VISIT         1011306 non-null  object        
 10  HOSPITAL         1011306 non-null  object        
 11  HOSPDAYS         52962 non-null    float64       
 12  X_STAY           1011306 non-null  object        
 13  DISABLE          1011306 non-null  object        
 14  RECOVD 

## Handle Non-Date Outliers

Code process adapted from: https://www.geeksforgeeks.org/data-analysis/detect-and-remove-the-outliers-using-python/

In [86]:
# get upper & lower values for outliers
upper_AGE = get_iqr_upper(df_VAERS_filtered['AGE_YRS'])
lower_AGE = get_iqr_lower(df_VAERS_filtered['AGE_YRS'])

upper_HOSPDAYS = get_iqr_upper(df_VAERS_filtered['HOSPDAYS'])
lower_HOSPDAYS = get_iqr_lower(df_VAERS_filtered['HOSPDAYS'])

upper_NUMDAYS = get_iqr_upper(df_VAERS_filtered['NUMDAYS'])
lower_NUMDAYS = get_iqr_lower(df_VAERS_filtered['NUMDAYS'])

print("Upper IQR value for AGE_YRS: ", upper_AGE)
print("Lower IQR value for AGE_YRS: ", lower_AGE)
print("Upper IQR value for HOSPDAYS: ", upper_HOSPDAYS)
print("Lower IQR value for HOSPDAYS: ", lower_HOSPDAYS)
print("Upper IQR value for NUMDAYS: ", upper_NUMDAYS)
print("Lower IQR value for NUMDAYS: ", lower_NUMDAYS)

# Code Example from: https://datagy.io/pandas-iqr/

Upper IQR value for AGE_YRS:  112.5
Lower IQR value for AGE_YRS:  0
Upper IQR value for HOSPDAYS:  14.5
Lower IQR value for HOSPDAYS:  0
Upper IQR value for NUMDAYS:  20.0
Lower IQR value for NUMDAYS:  0


In [87]:
# get indices of rows to drop

# AGE_YRS
index_AGE_YRS_upper = (df_VAERS_filtered.index[df_VAERS_filtered['AGE_YRS'] > upper_AGE]).tolist()
index_AGE_YRS_lower = (df_VAERS_filtered.index[df_VAERS_filtered['AGE_YRS'] < lower_AGE]).tolist()

# HOSPDAYS
index_HOSPDAYS_upper = (df_VAERS_filtered.index[df_VAERS_filtered['HOSPDAYS'] > upper_HOSPDAYS]).tolist()
index_HOSPDAYS_lower = (df_VAERS_filtered.index[df_VAERS_filtered['HOSPDAYS'] < lower_HOSPDAYS]).tolist()

# NUMDAYS
index_NUMDAYS_upper = (df_VAERS_filtered.index[df_VAERS_filtered['NUMDAYS'] > upper_NUMDAYS]).tolist()
index_NUMDAYS_lower = (df_VAERS_filtered.index[df_VAERS_filtered['NUMDAYS'] < lower_NUMDAYS]).tolist()

index_rows_to_drop = index_AGE_YRS_upper + index_AGE_YRS_lower + index_HOSPDAYS_upper + index_HOSPDAYS_lower + index_NUMDAYS_upper + index_NUMDAYS_lower

print(index_rows_to_drop[0:20])
print("Number of rows to drop: ", len(index_rows_to_drop))
print("Current length of dataframe: ", len(df_VAERS_filtered))
print("Length of dataframe after dropping rows: ", (len(df_VAERS_filtered) - len(index_rows_to_drop)))

[20982, 20989, 21040, 72228, 81988, 82014, 138032, 256843, 479003, 664407, 826845, 904031, 36492, 36580, 51524, 65392, 76878, 77113, 79972, 82652]
Number of rows to drop:  158848
Current length of dataframe:  1011306
Length of dataframe after dropping rows:  852458


In [88]:
# drop rows
df_VAERS_filtered = df_VAERS_filtered.drop(index_rows_to_drop)

In [89]:
# verify filter
df_VAERS_filtered.describe().T

,count,mean,min,25%,50%,75%,max,std
VAERS_ID,855196.0,1642446.490471,902418.0,1180031.5,1556817.5,2047933.25,2776336.0,535769.061963
RECVDATE,855196,2021-09-23 02:58:46.096942336,2020-12-15 00:00:00,2021-04-01 00:00:00,2021-08-04 00:00:00,2021-12-27 00:00:00,2024-06-28 00:00:00,NaN
AGE_YRS,756316.0,48.162268,0.08,33.0,49.0,64.0,110.0,19.990037
RPT_DATE,1099,2021-06-14 11:17:24.949953792,2020-01-13 00:00:00,2021-01-11 00:00:00,2021-02-18 00:00:00,2021-08-17 00:00:00,2024-05-21 00:00:00,NaN
DATEDIED,6267,2021-06-29 19:30:42.221158656,2020-01-10 00:00:00,2021-02-22 00:00:00,2021-04-08 00:00:00,2021-09-05 12:00:00,2024-06-22 00:00:00,NaN
HOSPDAYS,22319.0,3.76688,1.0,2.0,3.0,5.0,14.0,2.84879
VAX_DATE,781374,2021-07-08 22:19:57.244852992,2020-01-01 00:00:00,2021-02-19 00:00:00,2021-04-12 00:00:00,2021-10-04 00:00:00,2024-06-27 00:00:00,NaN
ONSET_DATE,758047,2021-07-10 04:05:16.267460608,2020-01-01 00:00:00,2021-02-24 00:00:00,2021-04-14 00:00:00,2021-10-03 00:00:00,2024-06-27 00:00:00,NaN
NUMDAYS,719253.0,1.985999,0.0,0.0,0.0,2.0,20.0,3.775713
FORM_VERS,855196.0,1.99863,1.0,2.0,2.0,2.0,2.0,0.036994


In [90]:
# verify filter
df_VAERS_filtered.info()

<class 'pandas.core.frame.DataFrame'>
Index: 855196 entries, 0 to 1012893
Data columns (total 28 columns):
 #   Column           Non-Null Count   Dtype         
---  ------           --------------   -----         
 0   VAERS_ID         855196 non-null  Int64         
 1   RECVDATE         855196 non-null  datetime64[ns]
 2   STATE            713367 non-null  object        
 3   AGE_YRS          756316 non-null  float64       
 4   SEX              855196 non-null  object        
 5   RPT_DATE         1099 non-null    datetime64[ns]
 6   DIED             855196 non-null  object        
 7   DATEDIED         6267 non-null    datetime64[ns]
 8   L_THREAT         855196 non-null  object        
 9   ER_VISIT         855196 non-null  object        
 10  HOSPITAL         855196 non-null  object        
 11  HOSPDAYS         22319 non-null   float64       
 12  X_STAY           855196 non-null  object        
 13  DISABLE          855196 non-null  object        
 14  RECOVD           855196 

# Save cleaned VAERS data to csv

In [91]:
# save cleaned VAERS dataframe to csv
path_to_save = path + '/VAERSDATA_cleaned.csv'
path_to_save

# save transformed symptoms dataframe to csv
df_VAERS_filtered.to_csv(path_to_save)